In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

ENDSWITH = "notebooks"

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDSWITH):
    raise ValueError(f"Not in correct dir, expect end with {ENDSWITH}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

In [2]:
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Keep runtime artifacts at repo root (.data/...), not under src/notebooks/
DATA_DIR = os.path.join(BASE_DIR, ".data")

In [5]:
from src.yt_crawler.YtCrawlerClass import YtCrawler

crawler = YtCrawler(
    output_dir=os.path.join(DATA_DIR, "yt_crawler", "downloads"),
    work_dir=os.path.join(DATA_DIR, "yt_crawler", "work"),
)

# # link = "https://youtu.be/u4b4VA7luW0?si=jRCJtpqjXPxVC-uX"
link = "https://youtu.be/a9aVQCWi9MY"

audio = crawler.ingest(link)
audio.quick_save()

# audio.save_to(os.path.join(BASE_DIR, "benchmarks/separation/sources/speech/vietcettera_clean.wav"))

Quick saved to: /home/tungnl5/Documents/audio-prepare-pipeline-redo/temp/a9aVQCWi9MY__89.63s_44100Hz_1ch.wav


Audio(source_id='a9aVQCWi9MY', title='Movie cut | Trích đoạn KHÁCH MỜI HẢI LAN: MỘT LẦN NÓI HẾT - THỎ ƠI!! - Phim Tết TRẤN THÀNH 2026', path='/home/tungnl5/Documents/audio-prepare-pipeline-redo/temp/a9aVQCWi9MY__89.63s_44100Hz_1ch.wav', sample_rate=44100, native_sample_rate=48000, duration_s=89.63s, channels=1, format='wav')

In [ ]:
from src.utils.AudioClass import Audio

# newaudio = Audio.from_file(BASE_DIR + "/benchmarks/separation/sources/speech/ready_or_not.wav")
newaudio = Audio.from_file("/home/tungnl5/Documents/audio-prepare-pipeline-redo/temp/-xc9WnrAhBE__1325.37s_44100Hz_1ch.wav")

from src.utils.AudioCutter import AudioCutter

cutter = AudioCutter(output_dir=".data/audio_cutter/out")

# raw timestamps (seconds)
newaudio = cutter.cut(newaudio, 0.0, 45.0)

# percentage of duration (0–100)
# newaudio = cutter.cut(newaudio, 0, 25, unit="percent")

In [ ]:

newaudio.notebook_display()

In [ ]:
newaudio.show_mel_spectrogram()

In [ ]:
from src.separation import HTDemucs, BSRoFormer, MelRoFormer, MVSepMDX23

# Swap backends here; all implement BaseSeparator.separate(Audio) -> Audio

# separator = HTDemucs(
#     device=device,
#     output_dir=os.path.join(DATA_DIR, "demucs", "out"),
#     work_dir=os.path.join(DATA_DIR, "demucs", "work"),
# )

# Default SW checkpoint is 6-stem (vocals/drums/bass/...). For speech + BGM,
# a vocals-only model usually leaks less bass into the vocal stem.
separator = BSRoFormer(
    device=str(device),
    output_dir=os.path.join(DATA_DIR, "bs_roformer", "out"),
    work_dir=os.path.join(DATA_DIR, "bs_roformer", "work"),
    # model="roformer-model-bs-roformer-vocals-revive-v3e-by-unwa",
)

# separator = MelRoFormer(
#     device=str(device),
#     output_dir=os.path.join(DATA_DIR, "mel_roformer", "out"),
#     work_dir=os.path.join(DATA_DIR, "mel_roformer", "work"),

# )

# separator = MVSepMDX23(
#     device=str(device),
#     output_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "out"),
#     work_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "work"),
#     repo_dir=os.path.join(DATA_DIR, "mvsep_mdx23", "repo"),
# )

separator.load()

cleaned_audio = separator.separate(newaudio)


In [ ]:
cleaned_audio.notebook_display()

In [ ]:
from src.utils.SpectrogramComparer import SpectrogramComparer

comparer = SpectrogramComparer(sample_rate=16000)
comparer.compare(newaudio, cleaned_audio)


In [ ]:
from src.utils.WaveformComparer import WaveformComparer

waveform_comparer = WaveformComparer(sample_rate=16000)
waveform_comparer.compare(newaudio, cleaned_audio)


In [ ]:
cleaned_audio.save_to(BASE_DIR + "/data/testing/yt_crawler/cleaned_mvsep_mdx23_audio.wav")
if hasattr(separator, "close"):
    separator.close()